In [ ]:
import torch

from onomatron.data import Vocab, load_names, make_sequences, split_names
from onomatron.models.transformer import TransformerName
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train

# Apple Silicon: use the integrated GPU via MPS when available (much faster
# than CPU for this model); fall back to CPU otherwise.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## Load names, split train/val/test, build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

## Encode names as full sequences (x, y)

In [ ]:
X, Y = make_sequences(train_names, vocab)
X_val, Y_val = make_sequences(val_names, vocab)
X_test, Y_test = make_sequences(test_names, vocab)
# Move everything to the device once: the training loop indexes these tensors
X, Y = X.to(device), Y.to(device)
X_val, Y_val = X_val.to(device), Y_val.to(device)
X_test, Y_test = X_test.to(device), Y_test.to(device)
print(
    f"{X.shape[0]} train / {X_val.shape[0]} val / {X_test.shape[0]} test"
    f" sequences of length {X.shape[1]}"
)

## Build Transformer (Vaswany et al. 2017)


In [ ]:
block_size = X.shape[1]
n_emb = 64
n_head = 4
n_layer = 2
seed = 42

model = TransformerName(
    vocab=vocab,
    block_size=block_size,
    n_emb=n_emb,
    n_head=n_head,
    n_layer=n_layer,
    seed=seed,
).to(device)
print(f"Transformer model has {model.n_parameters} parameters.")

## Find a maximum learning rate

In [ ]:
# Pick lr (minibatch, can take a while)
lrs = (10 ** torch.linspace(-4, -0.5, 100)).tolist()
losses = lr_range_test(
    lambda s: TransformerName(vocab, block_size, n_emb, n_head, n_layer, seed=s).to(
        device
    ),
    X,
    Y,
    lrs,
    batch_size=64,
)
best_lr = plot_lr_range(lrs, losses) / 10
print(f"Maximum learning rate found is {best_lr}")

## Train on train, watch val

In [ ]:
n_steps = 15000
batch_size = 64
lr = best_lr

opt = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=0.1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_steps)

history = train(
    model,
    X,
    Y,
    opt,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched,
    seed=seed,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,  # vanilla RNN: gradients can explode
    patience=20,
)
plot_history(history, smooth=20)
print(
    f"Final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
)

## Sample from the model (min 3 chars)

In [ ]:
model.cpu()  # sampling runs one char at a time
model.sample(20)

## Final evaluation on test

In [ ]:
model.to(device)
print(
    f"Transformer: train NLL {model.nll(X, Y):.4f}"
    f" | val NLL {model.nll(X_val, Y_val):.4f}"
    f" | test NLL {model.nll(X_test, Y_test):.4f}"
)